# 第13课 - 代理记忆


## 设置

本笔记本演示了如何使用**Microsoft Agent Framework**（MAF）构建具有**持久记忆**的旅行预订代理。

您将学习不同类型的代理记忆——工作记忆、短期记忆和长期记忆——如何影响代理在对话中保留和使用信息。

**先决条件：**
- 一个部署了聊天模型（例如 `gpt-4o-mini`）的 Azure AI Foundry 项目。
- 使用 Azure CLI 登录——在终端运行 `az login`。
- `AZURE_AI_PROJECT_ENDPOINT`——您的 Azure AI Foundry 项目终结点。
- `AZURE_AI_MODEL_DEPLOYMENT_NAME`——您已部署模型的名称。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## 代理记忆的类型

AI 代理可以利用不同类型的记忆，每种记忆都有其独特的用途：

### 工作记忆
对话线程本身——在单个会话中交换的消息。代理可以参考同一线程中较早的消息以保持连贯性。在 MAF 中，这通过 **`agent.create_session()`** 创建，返回一个 `AgentSession`。

### 短期记忆
在任务或会话期间持续存在但不会永久存储的信息。例如，代理可能在多轮规划对话中积累事实，并利用这些事实生成最终的行程。

### 长期记忆
跨 **多次会话** 持续存在的偏好和事实。回访用户不应重复他们的饮食限制或旅行风格。长期记忆通常由外部存储支持——数据库、文件或向量索引——并通过工具呈现给代理。


## 使用会话的工作记忆

最简单的记忆形式是会话。当你将同一个会话（通过 `agent.create_session()` 创建）传递给连续的 `agent.run()` 调用时，代理可以看到该对话的完整历史并回忆之前的细节。

让我们创建一个旅行代理并演示工作记忆。


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

代理正确地回忆了预算，因为两个消息共享同一个会话。这是**工作记忆**——它只存在于会话的生命周期内。

### 新线程会发生什么？

如果我们创建一个**新的**会话，代理将不记得之前的对话：


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## 长期记忆模式

为了记住用户在**跨会话**中的偏好，我们需要一个存在于对话线程之外的持久存储。代理通过**工具**访问该存储——这些工具是它可以调用以保存和检索信息的函数。

下面我们实现了一个简单的内存中偏好存储（在生产中，你会用数据库或向量索引来支持它）并将其作为代理可以使用的工具暴露出来。

### 架构
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### 情景 1 — 首次用户预订纪念日旅行

Sarah 第一次访问。代理应通过工具存储她的偏好，并利用这些偏好推荐酒店。


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 场景 2 — Sarah 几周后返回

Sarah 开始一个**全新的对话**（模拟一个新会话）。工作记忆是空的，但长期偏好存储中仍然有她的信息。代理应检索这些信息并用它来个性化推荐。


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## 总结

在本课中，您学习了三种类型的代理记忆以及如何使用 Microsoft Agent Framework 实现它们：

| Memory Type | MAF Mechanism | Lifetime |
|---|---|---|
| **工作记忆** | `agent.create_session()` | 单次对话 |
| **短期记忆** | 线程内累积的上下文 | 单个任务/会话 |
| **长期记忆** | 通过 `@tool` 函数访问的外部存储 | 跨会话 |

### 关键要点
1. **`agent.create_session()`** 提供工作记忆 —— 代理可以查看会话内的完整对话历史。
2. **新会话会丢失上下文** —— 没有长期记忆，代理无法回忆过去的对话。
3. **`@tool` 函数** 弥合了这一差距 —— 它们允许代理保存和检索持久存储的信息。
4. **个性化随着时间改善** —— 存储的偏好越多，代理的推荐就越准确。

### 真实世界应用
- **客户服务**：记住客户历史和偏好
- **个人助理**：跨天或周维持上下文
- **医疗健康**：跟踪患者信息和偏好
- **电子商务**：基于历史的个性化购物

### 后续步骤
- 用数据库或向量存储（如 Azure AI Search）替代内存字典
- 为时间敏感信息添加记忆过期机制
- 构建具有共享记忆的多代理系统
- 探索基于知识图谱记忆的 Cognee 笔记本


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)翻译。尽管我们力求准确，但请注意自动翻译可能存在错误或不准确之处。原始语言版本应被视为权威来源。对于重要信息，建议使用专业人工翻译。对于因使用本翻译而产生的任何误解或误释，我们不承担任何责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
